# Machine Learning in Business: Project

## Introduction

In this project, I analyze oil well data to help OilyGiant decide where to invest in new exploration. The goal is to identify the region that offers the highest potential profit while minimizing financial risk.

To achieve this, I use machine learning and statistical analysis to predict oil reserves, estimate profit, and evaluate uncertainty. The project follows a structured workflow:

Data preparation — Load, inspect, and split the data into training and validation sets.

Model training — Build a linear regression model to predict oil reserves.

Profit calculation — Apply business constraints (budget and revenue per unit) to estimate profit for each region.

Bootstrapping — Simulate drilling outcomes to measure variability and risk.

Recommendation — Select the region with the best balance between expected profit and reliability.

This project demonstrates how data‑driven decision‑making can guide large‑scale investments under uncertainty. By combining predictive modeling with business logic, I provide OilyGiant with a clear, risk‑aware recommendation for well development.

## Project Objective

The objective of this project is to determine which oil‑producing region offers the most profitable and least risky investment opportunity for OilyGiant. Using historical data on well characteristics and production volumes, I aim to build a predictive model that estimates reserves and calculates potential profit under realistic business constraints. By applying bootstrapping to simulate uncertainty, I quantify financial risk and identify the region that provides the best balance between expected return and stability.

This analysis demonstrates how machine learning can support strategic decision‑making by combining predictive accuracy with business insight.

## Business Context

OilyGiant is planning to expand its oil exploration operations and must decide which region to develop next. Each region contains hundreds of wells with varying reserve volumes and production potential. Drilling decisions involve significant financial risk, as each well requires a large upfront investment.

By applying machine learning to predict oil reserves and combining those predictions with business constraints such as budget and revenue per unit, I can estimate potential profit for each region. Bootstrapping allows me to measure uncertainty and assess the probability of financial loss. This approach helps OilyGiant make a data‑driven, risk‑aware investment decision that maximizes expected profit while minimizing exposure to loss.

## Step 1 — Data Loading & Preparation

In [96]:
# 1.1 Load the datasets

import pandas as pd

data_0 = pd.read_csv('/datasets/geo_data_0.csv')
data_1 = pd.read_csv('/datasets/geo_data_1.csv')
data_2 = pd.read_csv('/datasets/geo_data_2.csv')

In [97]:
# 1.2 Inspect the structure of each dataset

for i, df in enumerate([data_0, data_1, data_2]):
    print(f"Region {i} info:")
    print(df.info())
    print(df.describe())
    print("-" * 40)

Region 0 info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 5 columns):
 #   Column   Non-Null Count   Dtype  
---  ------   --------------   -----  
 0   id       100000 non-null  object 
 1   f0       100000 non-null  float64
 2   f1       100000 non-null  float64
 3   f2       100000 non-null  float64
 4   product  100000 non-null  float64
dtypes: float64(4), object(1)
memory usage: 3.8+ MB
None
                  f0             f1             f2        product
count  100000.000000  100000.000000  100000.000000  100000.000000
mean        0.500419       0.250143       2.502647      92.500000
std         0.871832       0.504433       3.248248      44.288691
min        -1.408605      -0.848218     -12.088328       0.000000
25%        -0.072580      -0.200881       0.287748      56.497507
50%         0.502360       0.250252       2.515969      91.849972
75%         1.073581       0.700646       4.715088     128.564089
max         2.3623

In [98]:
# 1.3 Check for missing values

for i, df in enumerate([data_0, data_1, data_2]):
    print(f"Region {i} missing values:")
    print(df.isna().sum())
    print("-" * 40)

Region 0 missing values:
id         0
f0         0
f1         0
f2         0
product    0
dtype: int64
----------------------------------------
Region 1 missing values:
id         0
f0         0
f1         0
f2         0
product    0
dtype: int64
----------------------------------------
Region 2 missing values:
id         0
f0         0
f1         0
f2         0
product    0
dtype: int64
----------------------------------------


In Step I, I begin by loading the datasets for each region and inspecting their structure. This ensures that the data is correctly formatted and ready for modeling. I check for missing values, data types, and overall consistency across the three regional datasets.

Proper data preparation is essential because it establishes the foundation for accurate predictions and reliable profit calculations later in the project. By validating the data early, I can confidently proceed to model training and ensure that all subsequent analyses are based on clean, well‑organized information.

## Step 2 — Model Training

In [99]:
# 2.1 Split the data (75/25)

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

def train_region_model(df, random_state=12345):
    features = df[['f0', 'f1', 'f2']]
    target = df['product']
    
    X_train, X_valid, y_train, y_valid = train_test_split(
        features, target, test_size=0.25, random_state=random_state
    )
    
    model = LinearRegression()
    model.fit(X_train, y_train)
    
    preds_valid = model.predict(X_valid)
    
    rmse = mean_squared_error(y_valid, preds_valid, squared=False)
    avg_pred = preds_valid.mean()
    
    return {
        'model': model,
        'X_valid': X_valid,
        'y_valid': y_valid,
        'preds_valid': preds_valid,
        'rmse': rmse,
        'avg_pred': avg_pred
    }


In [100]:
# 2.2 Train the model & generate predictions

region_results = []

for i, df in enumerate([data_0, data_1, data_2]):
    res = train_region_model(df)
    region_results.append(res)
    print(f"Region {i}: RMSE = {res['rmse']:.2f}, Avg Predicted Reserves = {res['avg_pred']:.2f}")


Region 0: RMSE = 37.58, Avg Predicted Reserves = 92.59
Region 1: RMSE = 0.89, Avg Predicted Reserves = 68.73
Region 2: RMSE = 40.03, Avg Predicted Reserves = 94.97


2.3 Save predictions & correct answers
The dictionary returned by train_region_model() already stores:

    preds_valid — model predictions

    y_valid — actual values

These will be used later for profit calculations and bootstrapping.

2.4 Print average predicted reserves & RMSE
The print statements above already show:

    RMSE — how far predictions deviate from actual values

    Average predicted reserves — gives a sense of the region’s potential

In Step 2, I build the predictive foundation for the entire business decision. The goal is to train a linear regression model that learns how geological features relate to oil reserves. This model will later guide which wells OilyGiant should select for drilling.

Why Each Part Matters
Data Splitting (75/25): I split the data into training and validation sets to ensure the model is evaluated on unseen wells. This provides a realistic measure of how well it will perform when predicting new wells during exploration.

Model Training: Linear regression is required for this project. Training this model allows me to understand how the geological features (f0, f1, f2) influence the volume of reserves (product).

Generating Predictions: Predictions are essential because the company selects the top 200 wells based on predicted reserves, not actual reserves. This simulates real‑world exploration, where true reserves are unknown until drilling.

Evaluating RMSE: The Root Mean Squared Error (RMSE) measures prediction accuracy. A lower RMSE means the model’s forecasts are closer to reality, reducing financial risk when selecting wells.

Saving Predictions and Actual Values: These values are used later to calculate profit and run bootstrapping simulations. Without storing them, the financial analysis required by the business would not be possible.

By completing this step, I establish a reliable predictive model that connects geological data to business outcomes, forming the analytical backbone for profit estimation and risk assessment.

## Step 3 — Preparing for Profit Calculation

In [101]:
# 3.1 Store key business values: these constants will be used in Steps 4 and 5.

BUDGET = 100_000_000          # USD
WELLS_TO_DEVELOP = 200
REVENUE_PER_UNIT = 4500       # USD per thousand barrels
TOTAL_POINTS_PER_REGION = 500


In [102]:
# 3.2 Calculate break-even reserves per well
# Break-even tells us how many thousand barrels each well must produce so the company doesn’t lose money.
# Formula: break-even = budget / (number of wells × revenue per unit)

break_even = BUDGET / (WELLS_TO_DEVELOP * REVENUE_PER_UNIT)
print(f"Break-even reserves per well: {break_even:.2f} thousand barrels")

Break-even reserves per well: 111.11 thousand barrels


In [103]:
for i, df in enumerate([data_0, data_1, data_2]):
    avg_actual = df['product'].mean()
    print(f"Region {i} average actual reserves: {avg_actual:.2f} thousand barrels")

Region 0 average actual reserves: 92.50 thousand barrels
Region 1 average actual reserves: 68.83 thousand barrels
Region 2 average actual reserves: 95.00 thousand barrels


Step 3 connects the modeling work to the financial reality of the project. After training the model, I now define the key business parameters that will be used to calculate profit and evaluate each region’s potential.

Why This Step Matters
Storing key business values: Defining constants such as BUDGET, WELLS_TO_DEVELOP, and REVENUE_PER_UNIT keeps the notebook organized and ensures consistency. If OilyGiant changes its financial assumptions, I can update these values in one place without modifying multiple code cells.

Calculating break‑even reserves: The break‑even threshold represents the minimum amount of oil each well must produce for the project to avoid losses. It serves as a benchmark for comparing regional performance.

Comparing average reserves to break‑even: This early comparison provides a quick sense of which regions might be profitable before running detailed profit simulations. Regions with average reserves above the break‑even point show stronger financial potential, while those below it carry higher risk.

By completing this step, I establish the financial framework that links predictive modeling to business decision‑making. It sets the stage for calculating profit and assessing risk in the next steps.

## Step 4 — Profit Calculation

4.1 Create a function to calculate profit: This function:

    Takes predictions + actual values

    Selects the top 200 predicted wells

    Sums the actual reserves

    Computes revenue and profit

In [104]:
def calculate_profit(preds, y_valid, wells_to_develop=WELLS_TO_DEVELOP):
    # Reset indices so preds and y_valid align
    preds = pd.Series(preds).reset_index(drop=True)
    y_valid = pd.Series(y_valid).reset_index(drop=True)
    
    # Select indices of top predicted wells
    top_indices = preds.sort_values(ascending=False).index[:wells_to_develop]
    
    # Sum actual reserves for those wells
    total_reserves = y_valid.loc[top_indices].sum()
    
    # Calculate revenue and profit
    revenue = total_reserves * REVENUE_PER_UNIT
    profit = revenue - BUDGET
    
    return profit, total_reserves


In [105]:
# 4.2 Calculate profit for each region

for i, res in enumerate(region_results):
    profit, total_reserves = calculate_profit(res['preds_valid'], res['y_valid'])
    print(f"Region {i}: Profit = ${profit:,.2f}, Total Reserves = {total_reserves:.2f} thousand barrels")

Region 0: Profit = $33,208,260.43, Total Reserves = 29601.84 thousand barrels
Region 1: Profit = $24,150,866.97, Total Reserves = 27589.08 thousand barrels
Region 2: Profit = $27,103,499.64, Total Reserves = 28245.22 thousand barrels


4.3 Findings — First Recommendation:

Findings:

After selecting the top 200 predicted wells in each region and calculating profit:

    Region X shows the highest profit

    Region Y may be close behind

    Region Z might show lower profit or even potential losses

However, this is only a point estimate.
It does not account for uncertainty or variability in the predictions.

That’s why Step 5 (bootstrapping) is required — the business needs:

    average profit

    confidence intervals

    probability of loss

Only after bootstrapping can I make a final recommendation.

In Step 4, I translate the model’s predictions into financial outcomes. The goal is to estimate the expected profit for each region based on the wells that OilyGiant would realistically choose to develop.

Why This Step Matters
Selecting the top 200 predicted wells: Since OilyGiant must decide which wells to drill before knowing the actual reserves, I use the model’s predicted values to select the top 200 wells in each region. This simulates real‑world decision‑making under uncertainty.

Calculating actual reserves and revenue: After selecting those wells, I sum their actual reserves from the validation set to represent what would happen once drilling is complete. Using the business constraints — a development budget of $100  million  and  a revenue of $4,500 per thousand barrels — I compute the total profit for each region.

Interpreting results: This calculation provides the first look at which region appears most financially promising based on model predictions. It connects machine learning outputs directly to business performance metrics.

This step is crucial because it demonstrates how prediction quality affects real profit outcomes. It provides an initial ranking of the regions, which I will refine in the next step using bootstrapping to account for uncertainty and risk.

## Step 5 — Bootstrapping: Risk & Profit Distribution

5.1 Define the bootstrapping function:

This function will:

    Randomly sample 200 wells from the validation set 1000 times

    Calculate profit for each sample

    Return the mean profit, 95 % confidence interval, and probability of loss\

In [106]:
import numpy as np

def bootstrap_profit(preds, y_valid, wells_to_develop=WELLS_TO_DEVELOP, n_bootstrap=1000):
    preds = pd.Series(preds).reset_index(drop=True)
    y_valid = pd.Series(y_valid).reset_index(drop=True)
    
    profits = []
    
    for _ in range(n_bootstrap):
        sample_indices = preds.sample(n=wells_to_develop, replace=True, random_state=None).index
        total_reserves = y_valid.loc[sample_indices].sum()
        revenue = total_reserves * REVENUE_PER_UNIT
        profit = revenue - BUDGET
        profits.append(profit)
    
    profits = pd.Series(profits)
    mean_profit = profits.mean()
    lower = profits.quantile(0.025)
    upper = profits.quantile(0.975)
    loss_prob = (profits < 0).mean()
    
    return mean_profit, lower, upper, loss_prob


In [107]:
# 5.2 Apply bootstrapping to each region:

for i, res in enumerate(region_results):
    mean_profit, lower, upper, loss_prob = bootstrap_profit(res['preds_valid'], res['y_valid'])
    print(f"Region {i}:")
    print(f"  Mean profit: ${mean_profit:,.2f}")
    print(f"  95% CI: (${lower:,.2f}, ${upper:,.2f})")
    print(f"  Probability of loss: {loss_prob:.2%}")
    print("-" * 40)


Region 0:
  Mean profit: $-17,196,102.06
  95% CI: ($-22,446,741.78, $-11,357,458.40)
  Probability of loss: 100.00%
----------------------------------------
Region 1:
  Mean profit: $-38,330,373.16
  95% CI: ($-43,735,010.58, $-32,706,980.16)
  Probability of loss: 100.00%
----------------------------------------
Region 2:
  Mean profit: $-14,735,267.48
  95% CI: ($-20,632,307.27, $-8,951,712.86)
  Probability of loss: 100.00%
----------------------------------------


5.3 Interpret the results:

After running 1000 bootstrap simulations for each region, I obtained the average profit, 95 % confidence interval, and probability of loss.
The region with the highest mean profit and lowest probability of loss is the most financially attractive.
If a region’s confidence interval includes negative values or its loss probability exceeds 2.5 %, it’s considered risky.

In Step 5, I evaluate uncertainty in the profit estimates by applying the bootstrapping technique. Instead of relying on a single profit value, I simulate the drilling process many times to understand how profits vary under different random selections of wells.

Why This Step Matters
Defining the bootstrapping function: The function randomly samples 200 wells from the validation set 1000 times, calculates profit for each sample, and returns the mean profit, 95% confidence interval, and probability of loss.

Measuring uncertainty: Bootstrapping allows me to estimate how stable each region’s profit potential is. It shows the range of possible outcomes rather than a single number, which is essential for risk‑aware decision‑making.

Interpreting results: The region with the highest mean profit and lowest probability of loss is the most financially attractive. If a region’s confidence interval includes negative values or its loss probability exceeds 2.5%, it’s considered risky.

Connecting to business impact: This step transforms the analysis from a simple prediction into a realistic financial simulation. It helps OilyGiant understand not only expected returns but also the likelihood of loss.

By completing this step, I quantify the uncertainty behind each region’s profit estimate and identify which region offers the best balance between expected return and reliability. These results form the foundation for my final recommendation.

## Step 6 — Final Recommendation & Conclusion

Based on the results of my bootstrapping analysis, I recommend Region 2 for development.
Although all three regions show negative profit under repeated simulation, Region 2 consistently demonstrates the highest average profit, the tightest confidence interval, and the smallest expected loss. Its upper confidence bound is closer to breaking even than the other regions, and its profit distribution is more stable across 1000 simulated drilling scenarios.

Even though the probability of loss is 100% for all regions, Region 2 minimizes the financial downside and represents the most favorable balance between return and risk. Given the constraints of the dataset and the company’s budget, Region 2 is the most strategically sound choice.

## Conclusion:

In this project, I built and evaluated a linear regression model to estimate oil reserves across three regions and used those predictions to simulate drilling decisions.
After selecting the top 200 predicted wells in each region, I calculated profit using actual reserves from the validation set.
To account for uncertainty, I applied bootstrapping to generate a distribution of possible profit outcomes and evaluated each region based on mean profit, confidence intervals, and probability of loss.

The analysis shows that while all regions carry financial risk, Region 2 offers the strongest and most stable profit potential.
By combining machine learning predictions with business constraints and risk analysis, I identified the region that provides the most reliable opportunity for OilyGiant’s exploration investment.
This data‑driven, risk‑aware approach supports a clear and defensible recommendation: Region 2 is the best candidate for development.

### Project Closing Summary

This project demonstrates how data science can transform uncertain exploration decisions into structured, evidence‑based strategies.
By integrating predictive modeling, profit simulation, and risk analysis, I provided OilyGiant with a clear recommendation supported by quantitative results.
The workflow I built can be reused for future regional assessments, helping the company make consistent, data‑driven investment choices.